In [ ]:
import pandas as pd

df= pd.read_csv('customer_shopping_behavior.csv')



In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df.isnull().sum()

In [ ]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
df.isnull().sum()

In [ ]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ' , '_')


In [ ]:
df= df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})


In [ ]:
df.columns

In [ ]:
labels = ['Young', 'Adult', 'Middle-aged', 'Senior']
df['age_group']= pd.qcut(df['age'], q=4 ,labels=labels)

In [ ]:
df[['age', 'age_group']].head(10)

In [ ]:
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days']=df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

In [ ]:
(df['discount_applied']== df['promo_code_used']).all()

In [ ]:
df= df.drop('promo_code_used', axis=1)

In [ ]:
df.columns

In [ ]:
%pip install pandas sqlalchemy pymysql

### Database connection
This project uses MySQL for the SQL analysis. Configure the connection in the MySQL cell below. Do not store passwords directly in a notebook or commit credentials to GitHub.

In [ ]:
# Load the cleaned DataFrame into MySQL.
table_name = "customer"

df.to_sql(
    name=table_name,
    con=engine,
    if_exists="replace",
    index=False,
)

print(f"Data successfully loaded into table '{table_name}'.")

In [ ]:
%pip install pymysql sqlalchemy

In [ ]:
import os
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Set these environment variables on your own computer before running this cell.
# Never commit database credentials to GitHub.
connection_url = URL.create(
    drivername="mysql+pymysql",
    username=os.environ.get("MYSQL_USER", "root"),
    password=os.environ["MYSQL_PASSWORD"],
    host=os.environ.get("MYSQL_HOST", "localhost"),
    port=int(os.environ.get("MYSQL_PORT", "3306")),
    database=os.environ.get("MYSQL_DATABASE", "customer_behavior"),
)

engine = create_engine(connection_url)

with engine.connect() as conn:
    print("MySQL connection successful!")

The cleaned DataFrame is loaded into the `customer` table in the preceding cell.

In [ ]:
pd.read_sql("SELECT * FROM customer LIMIT 5", engine)